# Projet : Les accidents corporels routiers en France 

## Introduction

Nous voulons étudier les accidents corporels en France sur différentes années. Pour ce faire nous avons récoltés les données issus de BAAC ... Nous avons pluisuers fichiers de données composés ... 
Nous avons 3 années : 2022, 2023, 2024

Problématique : Quels facteurs influencent la gravité des accidents corporels de la route en France, peut-on prédire cette gravité en tenant compte des caractéristiques de l'accident (environnementales, géographique...) ?

In [1]:
import matplotlib.pyplot as plt

# Traitement
from src.donnees import import_donnees, renomer_cle_jointure, concatenation_annees
from src.nettoyage import recodage, mapping_renommer_colonnes, colonnes_a_supprimer, création_age_usager, jointure, creation_mois_num, rajout_colonnes

# Analyse descriptive
from src.fct_carto import creation_df_carte, carte_departement

# Modelisation

In [2]:
import importlib
import src.nettoyage

importlib.reload(src.nettoyage)


<module 'src.nettoyage' from '/home/onyxia/work/Projet_pythonDS/src/nettoyage.py'>

## I - Traitement des données

Nous avons réalisé des analyses préliminaires sur nos données (disponible dans le fichier pre-processing), cela nous a permis de relever certains problèmes dans les données que nous avons pris en compte pour le nettoyage de nos données ci-dessous (comme les na, les noms de colonnes, les doublons ...)

In [3]:
donnees_completes = import_donnees()

donnees_completes["caract"][22] = renomer_cle_jointure(donnees_completes["caract"][22], "Num_Acc", "Accident_Id")

/home/onyxia/work/Projet_pythonDS/src/donnees.py:8: DtypeWarning: Columns (0: lartpc) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(nom_fichier_csv, sep=';', encoding='UTF-8')
/home/onyxia/work/Projet_pythonDS/src/donnees.py:8: DtypeWarning: Columns (0: nbv) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(nom_fichier_csv, sep=';', encoding='UTF-8')


In [4]:
# CARACTERISTIQUE
df_caract = concatenation_annees(donnees_completes, "caract")
df_caract = creation_mois_num(df_caract)
# LIEUX
df_lieux = concatenation_annees(donnees_completes, "lieux")
# VEHICULE
df_vehicule = concatenation_annees(donnees_completes, "vehicule")
# USAGER
df_usager = concatenation_annees(donnees_completes, "usager")

In [5]:
mappings = mapping_renommer_colonnes()

In [6]:
# recodage des noms des colonnes 
df_caract_recoder = recodage(df_caract, mappings["caract"])
df_lieux_recoder = recodage(df_lieux, mappings["lieux"])
df_vehicule_recoder = recodage(df_vehicule, mappings["vehicule"])
df_usager_recoder = recodage(df_usager, mappings["usager"])

# supprimer les doublons de corrections des données dans le fichier lieux 
df_lieux_recoder = df_lieux_recoder.drop_duplicates(subset="Num_Acc", keep="last")

# age pour les usagers
df_usager_recoder = création_age_usager(df_usager_recoder)

In [7]:
df_final = jointure(df_caract_recoder, df_lieux_recoder, df_vehicule_recoder, df_usager_recoder)

In [8]:
colonnes_a_supprimer = colonnes_a_supprimer()
df_final = df_final.drop(columns=colonnes_a_supprimer, errors="ignore")
df_final = df_final.dropna(subset=["id_usager"])

In [9]:
df_final = rajout_colonnes(df_final)

In [10]:
df_final.columns

Index(['Num_Acc', 'jour', 'mois', 'an', 'hrmn', 'lum', 'dep', 'agg', 'int',
       'atm', 'col', 'lat', 'long', 'mois_num', 'catr', 'circ', 'nbv', 'vosp',
       'prof', 'plan', 'surf', 'infra', 'situ', 'vma', 'id_vehicule', 'catv',
       'obs', 'obsm', 'choc', 'manv', 'id_usager', 'catu', 'grav', 'sexe',
       'trajet', 'secu1', 'age', 'date', 'jour_semaine', 'hr'],
      dtype='str')

In [11]:
df_final

,Num_Acc,jour,mois,an,hrmn,lum,dep,agg,int,atm,...,id_usager,catu,grav,sexe,trajet,secu1,age,date,jour_semaine,hr
0,202400000001,25,mars,2024,07:40,Crépuscule ou aube,70,Hors agglomération,Hors intersection,Brouillard - fumée,...,203 988 581,Conducteur,Blessé hospitalisé,Homme,Domicile - École,Ceinture,23,2024-03-25,Lundi,07
1,202400000001,25,mars,2024,07:40,Crépuscule ou aube,70,Hors agglomération,Hors intersection,Brouillard - fumée,...,203 988 582,Conducteur,Indemne,Homme,Utilisation professionnelle,Ceinture,29,2024-03-25,Lundi,07
2,202400000002,20,mars,2024,15:05,Plein jour,21,En agglomération,Intersection en T,Temps éblouissant,...,203 988 579,Piéton,Blessé hospitalisé,Femme,Promenade - loisirs,Aucun équipement,99,2024-03-20,Mercredi,15
3,202400000002,20,mars,2024,15:05,Plein jour,21,En agglomération,Intersection en T,Temps éblouissant,...,203 988 580,Conducteur,Indemne,Homme,Utilisation professionnelle,Ceinture,39,2024-03-20,Mercredi,15
4,202400000003,22,mars,2024,19:30,Crépuscule ou aube,15,Hors agglomération,Hors intersection,Normale,...,203 988 574,Passager,Blessé léger,Femme,Promenade - loisirs,Non déterminable,19,2024-03-22,Vendredi,19
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
377697,202200055301,1,janvier,2022,08:40,Plein jour,81,Hors agglomération,Intersection en T,Normale,...,968 230,Conducteur,Indemne,Femme,Promenade - loisirs,Ceinture,24,2022-01-01,Samedi,08
377698,202200055301,1,janvier,2022,08:40,Plein jour,81,Hors agglomération,Intersection en T,Normale,...,968 231,Passager,Blessé hospitalisé,Femme,Promenade - loisirs,Ceinture,22,2022-01-01,Samedi,08
377699,202200055301,1,janvier,2022,08:40,Plein jour,81,Hors agglomération,Intersection en T,Normale,...,968 232,Conducteur,Blessé léger,Femme,Promenade - loisirs,Ceinture,73,2022-01-01,Samedi,08
377700,202200055302,1,mars,2022,16:55,Plein jour,41,En agglomération,Hors intersection,Normale,...,968 228,Conducteur,Blessé hospitalisé,Homme,Domicile - Travail,Casque,34,2022-03-01,Mardi,16


Certaines variables contiennent des valeurs non renseigné, pour certaines nous décidons donc de supprimer les lignes comportant des valeurs pour d'autre nous enlevons la colonne.
Pour faire notre choix nous considérons plusieurs éléments :
- enlever les lignes : si les NA sont représneter en faible quantités dans la colonnes, si la colonne est importantes. Par exemple pour la colonne gravité de l'accidnet nous avons 359 NA sur 377 638 cela représente une faible proportion des usagers de plus la colonne gravité nous est totalement indispensable pour faire nos analyses car nous voulons regarder les gravités des différents accidents 
- enlever les colonnes : si il y a beaucoup de NA et que la colonne n'est pas forcement pertinentes pour nos analyses 

Les variables avaient une repartition assez fidèle sur la gravité des accidents. Sauf quelques variables qui n'avait que des indemnes ou tout sauf des tué, mais cela ne pose pas de pb car bcp dans les données. En revanche, pour la variable trajet (29% de NA), il y a trop de gravité Tué pour pouvoir supprimé les na de cette variable, nous la gardons pour l'analyser en revanche la modelisation ne la prendra pas en compte. De même poiur circ.

In [12]:
# suppression des NA 
df_final = df_final.dropna(subset=["lum", "int", "atm", "col", "prof", "plan", "surf", "situ", "catv", "obs", "obsm", "choc", "sexe", "age", "secu1"])

## II - Analyse Descriptive des accidents corporels en France

### 2. Évolution des accidents dans le temps

#### Vue d'ensemble

Il s'agit ici d'étudier les accidents sur la période 2022-2024 pour déterminer s'il y a eu une éventuelle augmentation ou diminution, et si les accidents présentent une cyclicité.

In [ ]:
nb_accidents_par(df_final, "an", "Année")

On remarque déjà que le nombre d'accidents est comparable sur les trois années. On peut ensuite s'intéresser à l'évolution mois par mois.

In [ ]:
evolution_mensuelle(df_final)

On constate d'une part que si à l'année le nombre d'accidents ne semble pas présenter de tendance, il varie d'un mois à l'autre en restant compris entre 3600 et 5300. D'autre part, le nombre d'accidents est généralement reflété par le nombre d'usagers impliqués (deux à trois usagers pour un accident en moyenne), lui-même reflété par le nombre de victimes (les usagers blessés ou tués). Les usagers indemnes représentent 41,23 % de ceux impliqués dans les accidents corporels de la base de données.

Évolutions hebdomadaire et horaire

In [ ]:
ordre = ["Lundi", "Mardi", "Mercredi", "Jeudi", "Vendredi", "Samedi", "Dimanche"]
nb_accidents_par(df_final, "jour_semaine", "Jour de la semaine", ordre, True)

In [ ]:
nb_accidents_par(df_final, "hr", "Heure")

Le vendredi est le jour de la semaine qui cumule le plus d'accidents (26804), et le dimanche le moins (20102). Le reste de la semaine comporte des valeurs aux alentours de 22000 ou 23000.

On observe deux modes aux heures de pointes, à 8 et 17 heures. Les accidents sont ensuite plus nombreux de jour que de nuit, et laissent supposer que leur nombre reflète au moins en partie l'intensité du trafic.

Les accidents sont d'ailleurs plus nocturnes le week-end que la semaine, et la distribution moins répartie autour des heures de pointes :

In [ ]:
nb_accidents_par(df_final[df_final["jour_semaine"].isin(["Samedi", "Dimanche"])], "hr", "Heure")

### 3. Nombre et gravité des accidents selon les facteurs extérieurs

#### Conditions atmosphériques et météorologiques

La majeure partie des accidents a lieu dans des conditions de conduite « normales » : sans défaut de visibilité dû à la météo ni problème d'éclairage, et sur des routes non altérées par l'environnement. La rareté des phénomènes inhabituels se retranscrit par une faible occurrence dans les données. Mises à part les conditions d'éclairage, les conditions qui dépendent directement ou indirectement de l'environnement sont de l'ordre des centaines seulement lorsqu'elles sortent de l'ordinaire.

In [ ]:
ordre_lignes_lum  = [
    "Plein jour",
    "Crépuscule ou aube",
    "Nuit sans éclairage public",
    "Nuit avec éclairage public non allumé",
    "Nuit avec éclairage public allumé"
]

nb_accidents_par(df_final, "lum", "Condition d'éclairage", ordre_lignes_lum, True)

In [ ]:
ordre_lignes_atm = [
    "Normale", 
    "Pluie légère", 
    "Pluie forte", 
    "Neige - grêle", 
    "Brouillard - fumée", 
    "Vent fort - tempête", 
    "Temps éblouissant", 
    "Temps couvert", 
    "Autre"
]
nb_accidents_par(df_final, "atm", "Condition atmosphérique", ordre_lignes_atm, True)

Il est en outre possible de s'intéresser à la gravité des accidents pour ces différentes modalités :

In [ ]:
tc_lum_grav = tab_cont_grav(df_final, "lum", ordre_lignes, ordre_colonnes)
bar_chart(tc_lum_grav, "Conditions d'éclairage", "Répartition de la gravité de l'accident selon la luminosité")

In [ ]:
tc_atm_grav = tab_cont_grav(df_final, "atm", ordre_lignes, ordre_colonnes)
bar_chart(tc_atm_grav, "Conditions atmosphériques", "Répartition de la gravité de l'accident selon les conditions atmosphériques")

Les situations où la visibilité est affectée semblent être les plus fatales. 6,3 % des usagers impliqués dans un accident de nuit sans éclairage public ont trouvé la mort (4,7 % pour les accidents avec éclairage public non allumé). Ce chiffre n'est que de 1,6 % avec ledit éclairage allumé, et de 2,3 % en plein jour.

De surcroît, le brouillard et/ou la fumée constituent la modalité associée à la plus forte mortalité parmi les conditions atmosphériques directes avec 6,4 % de tués.

In [ ]:
ordre_lignes_surf = [
   "Normale",
   "Mouillée",
   "Flaques",
   "Inondée",
   "Enneigée",
   "Boue",
   "Verglacée",
   "Corps gras - Huile",
   "Autre"
]

tc_surf_grav = tab_cont_grav(df_final, "surf", ordre_lignes_surf, ordre_colonnes)
bar_chart(tc_surf_grav, "Surface au sol", "Répartition de la gravité de l'accident selon la surface")

Bien que ces deux surfaces concernent un faible nombre d'accidents dans les données, la proportion de tués est plus grande lorsque les accidents ont lieu sur des surfaces inondées (87 accidents) ou boueuses (143) par rapport aux autres situations : 9,6 % et 7,6 % respectivement.

La part de blessés hospitalisés est particulièrement importante dans ce second cas (36,4 %, soit 10 points de plus que les autres types de surface spécifiés).

### 4. Dans quel département français les accidents sont-ils les plus graves ? Un peu de cartographie

Nous souhaitons voir si certaines régions (ou departements) ont plus d'accidents grave ... ?
Par accident, il peut y avoir plusiuers victime, nous considérons donc le nombre de victime et non le nombre d'accident 

In [ ]:
# nombre d'accident
print("nombre d'accidents : ", df_final["Num_Acc"].nunique())
# nombre de victime 
print("nombre de victimes : ", df_final["id_usager"].nunique())

In [ ]:
# recupération du nombre de victime par departement et gravité
df_victime_dep = df_final.groupby("dep").size().reset_index(name="nb").sort_values(by="nb")
print(df_victime_dep)

comme on peut le voir ci-dessus, le nombre d'accident par departement est très inégale, on observe notamment beaucoup d'accidnet dans les départemetn en ile de France, cela peut s'expiquer du fait qu'il ya plus d'habitants aussi 
Du a cela, nous allons étudier la proportion des blessures d'accident plutot que le nombre qui est trop disproportionné en fonction des département.

In [ ]:
print(df_final["grav"].unique())

In [ ]:
df_victimes_tot = creation_df_carte(df_final)

In [ ]:
carte_departement(df_victimes_tot)

In [ ]:
df_victimes_blessure = creation_df_carte(df_final, blessure=True)

In [ ]:
carte_departement(df_victimes_blessure, blessure="Tué")

In [ ]:
carte_departement(df_victimes_blessure, blessure="indemne")

Les victimes tués par un accdient sont une faible proportion des victimes, ou on en voit le plus dans la diagonal du vide 
En revanche, les accidents indemne sont plus vers le nord 
comparaison année à faire 

In [ ]:
df_victimes_an = creation_df_carte(df_final, an=True)

In [ ]:
carte_departement(df_victimes_an, an=2023)

In [ ]:
df_victimes_an_blessure = creation_df_carte(df_final, an=True, blessure=True)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(24, 8))

carte_departement(df_victimes_an_blessure, blessure="tué", an=2022, ax=axes[0])
carte_departement(df_victimes_an_blessure, blessure="tué", an=2023, ax=axes[1])
carte_departement(df_victimes_an_blessure, blessure="tué", an=2024, ax=axes[2])

fig.suptitle("Comparaison des tués par département – 2022 / 2023 / 2024", fontsize=18)
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(24, 8))

carte_departement(df_victimes_an_blessure, blessure="Blessé hospitalisé", an=2022, ax=axes[0])
carte_departement(df_victimes_an_blessure, blessure="Blessé hospitalisé", an=2023, ax=axes[1])
carte_departement(df_victimes_an_blessure, blessure="Blessé hospitalisé", an=2024, ax=axes[2])

fig.suptitle("Comparaison des Blessé hospitalisé par département – 2022 / 2023 / 2024", fontsize=18)
plt.tight_layout()
plt.show()


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(24, 8))

carte_departement(df_victimes_an_blessure, blessure="Blessé léger", an=2022, ax=axes[0])
carte_departement(df_victimes_an_blessure, blessure="Blessé léger", an=2023, ax=axes[1])
carte_departement(df_victimes_an_blessure, blessure="Blessé léger", an=2024, ax=axes[2])

fig.suptitle("Comparaison des Blessé léger par département – 2022 / 2023 / 2024", fontsize=18)
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(24, 8))

carte_departement(df_victimes_an_blessure, blessure="Indemne", an=2022, ax=axes[0])
carte_departement(df_victimes_an_blessure, blessure="Indemne", an=2023, ax=axes[1])
carte_departement(df_victimes_an_blessure, blessure="Indemne", an=2024, ax=axes[2])

fig.suptitle("Comparaison des Indemne par département – 2022 / 2023 / 2024", fontsize=18)
plt.tight_layout()
plt.show()